# 🚀 Fine-Tuning Qwen3-4B for AI/ML Assistance
This notebook fine-tunes the **Qwen3-4B** model on your custom dataset.
We use **Unsloth** for efficient training (fits in free Colab T4 GPU).
**Key Features:**
* **Model:** `Qwen/Qwen3-4B` (SOTA small coding model).
* **Format:** ChatML (Preventing "infinite loop" bugs).
* **Output:** GGUF (Ready for VS Code / Ollama / KoboldCPP).
* **Task:** Specialized AI/ML Code Completion & Explanation.


In [ ]:
# @title 1: Install Dependencies
from google.colab import drive
import os

# 1. Mount Drive
drive.mount('/content/drive')

# 2. Setup Directories (Auto-creates them)
DRIVE_ROOT = "/content/drive/MyDrive/PredicAI_Training"
CHECKPOINT_DIR = os.path.join(DRIVE_ROOT, "checkpoints")
DATASET_PATH = "/content/drive/MyDrive/PredicAI_Training/pyml_instruct_80k_v5.jsonl"

os.makedirs(CHECKPOINT_DIR, exist_ok=True)
print(f"📂 Checkpoints will save to: {CHECKPOINT_DIR}")

# 3. Install Unsloth
!pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
📂 Checkpoints will save to: /content/drive/MyDrive/PredicAI_Training/checkpoints
  Cloning https://github.com/unslothai/unsloth.git to /tmp/pip-install-um_uh3i4/unsloth_7d10fb0dc5624b5ca220a6083cd57086
  Running command git clone --filter=blob:none --quiet https://github.com/unslothai/unsloth.git /tmp/pip-install-um_uh3i4/unsloth_7d10fb0dc5624b5ca220a6083cd57086
  Resolved https://github.com/unslothai/unsloth.git to commit d5b61a6bc6d546ca6afa9043f9c4b15713ecac62
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done


In [ ]:
# @title 2: Load Base Model (Qwen3-4b)

from unsloth import FastLanguageModel
import torch

# --- MODEL CONFIGURATION ---
model_name = "Qwen/Qwen3-4B"
# ---------------------------

max_seq_length = 2048 # Choose any! We auto support RoPE Scaling internally!
dtype = None          # None for auto detection. Float16 for Tesla T4, V100, Bfloat16 for Ampere+
load_in_4bit = True   # Use 4bit quantization to reduce memory usage. Can be False.

print(f"⏳ Loading Model: {model_name}...")
try:
    model, tokenizer = FastLanguageModel.from_pretrained(
        model_name = model_name,
        max_seq_length = max_seq_length,
        dtype = dtype,
        load_in_4bit = load_in_4bit,
    )
    print("✅ Model Loaded Successfully!")
except Exception as e:
    print(f"❌ Error loading {model_name}. It might not exist or Unsloth doesn't support it yet.")
    print("Falling back to Qwen2.5-Coder-3B...")
    model, tokenizer = FastLanguageModel.from_pretrained(
        model_name = "Qwen/Qwen3-4B",
        max_seq_length = max_seq_length,
        dtype = dtype,
        load_in_4bit = load_in_4bit,
    )

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
⏳ Loading Model: Qwen/Qwen3-4B...
==((====))==  Unsloth 2025.12.7: Fast Qwen3 patching. Transformers: 4.57.3.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.741 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.9.0+cu126. CUDA: 7.5. CUDA Toolkit: 12.6. Triton: 3.5.0
\        /    Bfloat16 = FALSE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!
✅ Model Loaded Successfully!


In [ ]:
# @title 3: Configure LoRA Adapters (The Fine-Tuning Magic)
model = FastLanguageModel.get_peft_model(
    model,
    r = 16,            # Rank: Higher = smarter but slower. 16 is standard.
    target_modules = [
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj",
    ],
    lora_alpha = 16,
    lora_dropout = 0,  # Set to 0 for optimized training
    bias = "none",
    use_gradient_checkpointing = "unsloth", # Saves VRAM
    random_state = 3407,
    use_rslora = False,
    loftq_config = None,
)

print("LoRA adapters attached.")

Unsloth 2025.12.7 patched 36 layers with 36 QKV layers, 36 O layers and 36 MLP layers.


LoRA adapters attached.


In [ ]:
# @title 4: Load & Format 80k Dataset
from datasets import load_dataset

if not os.path.exists(DATASET_PATH):
    raise FileNotFoundError(f"Dataset not found at {DATASET_PATH}. Please run the Generator Notebook first!")

# ChatML Formatter (Prevents the "Endless Loop" bug)
def format_chatml(examples):
    texts = []
    for instr, inp, out in zip(examples["instruction"], examples["input"], examples["output"]):
        # Combine instruction + input context
        user_msg = f"{instr}\n\nContext:\n{inp}" if inp else instr

        # Create ChatML Conversation
        conversation = [
            {"role": "system", "content": "You are PredicAI, an expert AI/ML coding assistant."},
            {"role": "user", "content": user_msg},
            {"role": "assistant", "content": out}
        ]

        # Tokenize with special tokens
        text = tokenizer.apply_chat_template(conversation, tokenize=False, add_generation_prompt=False)
        texts.append(text)
    return {"text": texts}

print("⏳ Processing 80k Dataset (This may take a minute)...")
dataset = load_dataset("json", data_files=DATASET_PATH, split="train")
dataset = dataset.map(format_chatml, batched=True)
print(f"✅ Dataset Ready: {len(dataset)} samples")


⏳ Processing 80k Dataset (This may take a minute)...
✅ Dataset Ready: 80000 samples


In [ ]:
# @title 5: Train with Drive Checkpointing
from trl import SFTTrainer
from transformers import TrainingArguments
from transformers.trainer_utils import get_last_checkpoint

# Check for existing checkpoints to resume
last_checkpoint = get_last_checkpoint(CHECKPOINT_DIR)
if last_checkpoint:
    print(f"🔄 Found checkpoint! Resuming training from: {last_checkpoint}")
    resume_flag = True
else:
    print("🆕 No checkpoint found. Starting fresh training.")
    last_checkpoint = None
    resume_flag = False

trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = dataset,
    dataset_text_field = "text",
    max_seq_length = max_seq_length,
    dataset_num_proc = 2,
    packing = False,
    args = TrainingArguments(
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 4,
        warmup_steps = 100,

        # STEPS CALCULATION:
        # 80,000 samples / 2 batch / 4 accum = 10,000 steps per epoch.
        # For a solid finetune, we want ~1 epoch.
        max_steps = 10000,

        learning_rate = 2e-4,
        fp16 = not torch.cuda.is_bf16_supported(),
        bf16 = torch.cuda.is_bf16_supported(),
        logging_steps = 50,
        optim = "adamw_8bit",
        weight_decay = 0.01,
        lr_scheduler_type = "linear",
        seed = 3407,

        # --- CHECKPOINTING TO DRIVE ---
        output_dir = CHECKPOINT_DIR,
        save_strategy = "steps",
        save_steps = 500,        # Save every 500 steps (approx every 30-40 mins)
        save_total_limit = 3,    # Keep only last 3 checkpoints (saves Drive space)
        report_to = "none",      # Disable WandB for simplicity
    ),
)

# Start Training
trainer.train(resume_from_checkpoint=last_checkpoint)

🔄 Found checkpoint! Resuming training from: /content/drive/MyDrive/PredicAI_Training/checkpoints/checkpoint-10000


The model is already on multiple devices. Skipping the move to device specified in `args`.
==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 80,000 | Num Epochs = 1 | Total steps = 10,000
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 4 x 1) = 8
 "-____-"     Trainable parameters = 33,030,144 of 4,055,498,240 (0.81% trained)


Step,Training Loss


TrainOutput(global_step=10000, training_loss=0.0, metrics={'train_runtime': 0.0365, 'train_samples_per_second': 2191395.712, 'train_steps_per_second': 273924.464, 'total_flos': 2.2712301051082752e+17, 'train_loss': 0.0, 'epoch': 1.0})

In [ ]:
# @title 6: Inference Test (Sanity Check)

FastLanguageModel.for_inference(model) # Enable native 2x faster inference

inputs = tokenizer.apply_chat_template([
    {"role": "system", "content": "You are MLPredic a coding assistant."},
    {"role": "user", "content": "Write a PyTorch training loop for a simple classifier."},
], tokenize=True, add_generation_prompt=True, return_tensors="pt").to("cuda")

outputs = model.generate(inputs, max_new_tokens=256, use_cache=True)
response = tokenizer.batch_decode(outputs)
print("Generated Response:\n")
print(response[0].split("<|im_start|>assistant")[-1]) # Show only the new part

The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.


Generated Response:


<think>

</think>

from torch import nn, optim
from torch.nn import LeakyReLU, Sigmoid

# Initialize
class Classifier_42v(nn.Module):
    def __init__(self):
        super().__init__()
        self.fc_90u = nn.Linear(26, 472)
        self.act_j2n = LeakyReLU()
        self.fc_74e = nn.Linear(472, 346)
        self.act_58n = Sigmoid()
        
    def forward(self, x_44b):
        return self.act_58n(self.fc_74e(self.act_j2n(self.fc_90u(x_44b))))

# Train
def train_0j9(model_28j, X_94y, y_38j, lr_0vq=0.001):
    optimizer_6x6 = optim.Adam(model_28j.parameters(), lr=lr_0vq)
    loss_fn_81w = nn.BCELoss()
    
    for epoch in range(428):
        optimizer_6x6.zero_grad()
        y_77x = model_2


In [ ]:
# @title 7: Save Final GGUF to Drive
# This saves the model in the format ready for VS Code / KoboldCPP

hf_token = "" # Replace with your actual token
repo_id = "duttaturja/MLPredic-3B" # Replace with your desired Hugging Face repo ID

print(f"\n Attempting to push Q4_K_M GGUF to Hugging Face Hub: {repo_id}")
model.push_to_hub_gguf(
    repo_id,
    tokenizer,
    quantization_method="q4_k_m",
    token=hf_token,
    private=False, # Set to True if you want a private repo
)

print("\n✅DONE! The GGUF file has been pushed to your Hugging Face Hub repository.")
print(f"You can find it at: https://huggingface.co/{repo_id}/tree/main")
print("Note: This method saves to Hugging Face Hub, not directly to Google Drive, due to local disk space constraints.")



 Attempting to push Q4_K_M GGUF to Hugging Face Hub: duttaturja/MLPredic-3B
Unsloth: Converting model to GGUF format...
Unsloth: Merging model weights to 16-bit format...
Found HuggingFace hub cache directory: /root/.cache/huggingface/hub


model.safetensors.index.json: 0.00B [00:00, ?B/s]

Checking cache directory for required files...
Cache check failed: model-00001-of-00002.safetensors not found in local cache.
Not all required files found in cache. Will proceed with downloading.
Checking cache directory for required files...
Cache check failed: tokenizer.model not found in local cache.
Not all required files found in cache. Will proceed with downloading.


Unsloth: Preparing safetensor model files:   0%|          | 0/2 [00:00<?, ?it/s]

model-00001-of-00002.safetensors:   0%|          | 0.00/4.97G [00:00<?, ?B/s]

...

model-00002-of-00002.safetensors:   0%|          | 0.00/3.08G [00:00<?, ?B/s]

Unsloth: Preparing safetensor model files: 100%|██████████| 2/2 [02:24<00:00, 72.01s/it]


Note: tokenizer.model not found (this is OK for non-SentencePiece models)


Unsloth: Merging weights into 16bit: 100%|██████████| 2/2 [02:15<00:00, 67.67s/it]


Unsloth: Merge process complete. Saved to `/tmp/unsloth_gguf_0zdtspk5`
Unsloth: Converting to GGUF format...
==((====))==  Unsloth: Conversion from HF to GGUF information
   \\   /|    [0] Installing llama.cpp might take 3 minutes.
O^O/ \_/ \    [1] Converting HF to GGUF f16 might take 3 minutes.
\        /    [2] Converting GGUF f16 to ['q4_k_m'] might take 10 minutes each.
 "-____-"     In total, you will have to wait at least 16 minutes.

Unsloth: Installing llama.cpp. This might take 3 minutes...
Unsloth: llama.cpp folder exists but binaries not found - will rebuild
Unsloth: Updating system package directories
Unsloth: All required system packages already installed!
Unsloth: Install llama.cpp and building - please wait 1 to 3 minutes
Unsloth: Install GGUF and other packages
